# PFAS × ACS — 3D Mapping (pydeck)

Builds an interactive choropleth (shaded by an ACS variable) + 3D bars (PFAS reading) for a
chosen **month**, and writes a standalone HTML.

**Why a prep step:** the raw TIGER ZCTA shapefile is ~500 MB of full-resolution boundaries for
all ~33k ZCTAs. We never carry that into the repo or the HTML. Step 0 runs **once**: filter to
the ZCTAs we actually have data for, simplify, and save a small GeoParquet. That small file
(a few MB) is all the map needs — keeping both the repo and the exported HTML light.

In [120]:
import os, json, glob, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import pydeck as pdk
import geopandas as gpd


DATA  = Path(os.environ.get("DATA",  "../data/processed/pfas_acs_zcta_month.parquet"))
TIGER = Path(os.environ.get("TIGER", "../data/raw/tiger/cb_2020_us_zcta520_500k.shp"))  # raw download (gitignored)
GEOM  = Path(os.environ.get("GEOM",  "../data/processed/zcta_simplified.parquet"))      # small, committed
OUTLINE = Path(os.environ.get("OUTLINE", "../data/processed/zcta_outline_all.parquet"))
OUTDIR= Path(os.environ.get("OUTDIR","../maps")); OUTDIR.mkdir(parents=True, exist_ok=True)

## 0. One-time geometry prep (run once)

Reads the big shapefile, keeps only ZCTAs present in your data, reprojects to WGS84, simplifies,
and saves `zcta_simplified.parquet`. Skips automatically if that file already exists.

**Memory:** download the **cartographic boundary** file (`cb_2020_us_zcta520_500k`) rather than the
full `tl_2020_us_zcta520` — it's the pre-generalized version and far lighter to read. If even that
strains Colab, switch to the High-RAM runtime for this one cell; afterwards you only ever load the
small GeoParquet.

In [122]:
if GEOM.exists():
    print("geometry already prepared:", GEOM)
else:
    keep = set(pd.read_parquet(DATA, columns=["zcta"])["zcta"].astype(str).str.zfill(5))
    g = gpd.read_file(TIGER)                                   # uses pyogrio (fast, lighter)
    zcol = next(c for c in g.columns if c.upper().startswith("ZCTA5CE"))
    g = g[[zcol, "geometry"]].rename(columns={zcol: "ZCTA"})
    g["ZCTA"] = g["ZCTA"].astype(str).str.zfill(5)
    g = g[g["ZCTA"].isin(keep)].to_crs(4326)                   # filter FIRST, then reproject
    g["geometry"] = g.simplify(0.01, preserve_topology=True) 
    g.to_parquet(GEOM)
    print(f"wrote {GEOM}  ({GEOM.stat().st_size/1e6:.1f} MB, {len(g):,} ZCTAs)")

wrote ..\data\processed\zcta_simplified.parquet  (5.2 MB, 18,191 ZCTAs)


## 0b. One-time ALL-ZCTA outline prep (run once)

Writes `zcta_outline_all.parquet` — every US ZCTA (coarsely simplified), keeping the `zcta`
code. `make_map` uses it as a base layer so **every** area is shaded: ZCTAs with no data are
filled a distinct pink and read "Never sampled / Data Missing". Skips if the file exists.

In [129]:
if OUTLINE.exists():
    print("outline already prepared:", OUTLINE)
else:
    g = gpd.read_file(TIGER)
    zcol = next(c for c in g.columns if c.upper().startswith("ZCTA5CE"))
    g = g[[zcol, "geometry"]].rename(columns={zcol: "zcta"})
    g["zcta"] = g["zcta"].astype(str).str.zfill(5)
    g = g.to_crs(4326)
    g["geometry"] = g.simplify(0.02, preserve_topology=True)   # coarse — just fill + outline
    g.to_parquet(OUTLINE)
    print(f"wrote {OUTLINE}  ({OUTLINE.stat().st_size/1e6:.1f} MB, {len(g):,} ZCTAs)")

wrote ..\data\processed\zcta_outline_all.parquet  (5.6 MB, 33,791 ZCTAs)


## 1. Load data + geometry, merge

In [130]:
df  = pd.read_parquet(DATA)
df["zcta"] = df["zcta"].astype(str).str.zfill(5)

geo = gpd.read_parquet(GEOM)
geo["ZCTA"] = geo["ZCTA"].astype(str).str.zfill(5)
geo = geo.rename(columns={"ZCTA": "zcta"})[["zcta", "geometry"]]   # avoid clash w/ df's ACS 'ZCTA'

gdf = geo.merge(df, on="zcta", how="inner")
gdf = gpd.GeoDataFrame(gdf, geometry="geometry", crs=geo.crs)
# centroids for the 3D bars (projected CRS -> accurate, no warning), back to lon/lat
cent = gdf.to_crs(5070).geometry.centroid.to_crs(4326)
gdf["lon"], gdf["lat"] = cent.x.values, cent.y.values

ACS_VARS = [c for c in ["total_pop","median_hh_income","pct_below_poverty","pct_nonwhite",
                        "pct_hispanic","pct_black_nh","pct_asian_nh","pop_density"] if c in gdf.columns]
months = sorted(gdf["month"].dropna().unique())
print(f"{gdf['zcta'].nunique():,} ZCTAs with geometry | months: {len(months)} "
      f"({pd.Timestamp(months[0]):%Y-%m} … {pd.Timestamp(months[-1]):%Y-%m})")
print("available ACS variables:", ACS_VARS)

# all-ZCTA base for the "never sampled" pink fill (prepared in step 0b)
outline_gdf = gpd.read_parquet(OUTLINE)
outline_gdf["zcta"] = outline_gdf["zcta"].astype(str).str.zfill(5)
outline_gdf = gpd.GeoDataFrame(outline_gdf[["zcta", "geometry"]], geometry="geometry", crs=outline_gdf.crs)
print(f"outline ZCTAs (all US): {len(outline_gdf):,}")

18,191 ZCTAs with geometry | months: 36 (2023-01 … 2025-12)
available ACS variables: ['total_pop', 'median_hh_income', 'pct_below_poverty', 'pct_nonwhite', 'pct_hispanic', 'pct_black_nh', 'pct_asian_nh', 'pop_density']
outline ZCTAs (all US): 33,791


## 2. Blue → orange color ramp

In [131]:
BLUE, MID, ORANGE = np.array([74,125,224.]), np.array([150,150,170.]), np.array([224,114,74.])
def ramp(t):
    t = np.clip(np.nan_to_num(np.asarray(t, float), nan=0.0), 0, 1)
    lo = t < 0.5
    out = np.empty((t.size, 3))
    out[lo]  = BLUE + (MID-BLUE)   * (t[lo]/0.5)[:,None]
    out[~lo] = MID  + (ORANGE-MID) * ((t[~lo]-0.5)/0.5)[:,None]
    return out.round().astype(int)

def norm(s, lo_q=0.05, hi_q=0.95):
    s = pd.to_numeric(s, errors="coerce")
    lo, hi = s.quantile(lo_q), s.quantile(hi_q)
    hi = hi if hi > lo else lo + 1
    return ((s - lo) / (hi - lo)).clip(0, 1)

## 3. The map function

`make_map(month, acs_var)` → choropleth shaded by `acs_var` + 3D bars whose height **and** color
encode the mean PFAS reading, both on the blue→orange scale. Returns the pydeck `Deck` and writes
a standalone HTML.

> **Note**: Month parameter is left as future work. Currently it will take up too much memory.

In [132]:
ACS_LABELS = {
    "total_pop": "Total population", "median_hh_income": "Median household income ($)",
    "pct_below_poverty": "% below poverty", "pct_nonwhite": "% people of color",
    "pct_hispanic": "% Hispanic", "pct_black_nh": "% Black (NH)", "pct_asian_nh": "% Asian (NH)",
    "pop_density": "Population density (/sq mi)",
}
NA_PINK = [230, 175, 200]   # distinct "no data" pink: never-sampled areas + missing values

def make_map(month, acs_var, gdf=gdf, outline=None, elevation_max=180000, radius=1300, alpha=185,
             n_hotspots=12, layers="both", save=False):
    if acs_var not in gdf.columns:
        raise ValueError(f"{acs_var!r} not in data. Options: {ACS_VARS}")
    if layers not in ("both", "choropleth", "bars"):
        raise ValueError("layers must be 'both', 'choropleth', or 'bars'")
    if outline is None:
        outline = outline_gdf   # all-ZCTA base prepared in step 0b

    def _union_names(s):   # combine "; "-joined PWS names across a ZCTA's months
        out = set()
        for v in s.dropna():
            out.update(p.strip() for p in str(v).split(";") if p.strip())
        return "; ".join(sorted(out))

    # snapshot: all-time (one row per ZCTA) or a single month
    if month is None:
        ak = {"mean_result_ngL": ("mean_result_ngL", "mean"), acs_var: (acs_var, "first")}
        for col in ("city", "state"):
            if col in gdf.columns:
                ak[col] = (col, "first")
        if "pws_names" in gdf.columns:
            ak["pws_names"] = ("pws_names", _union_names)
        agg = gdf.groupby("zcta").agg(**ak).reset_index()
        base = gdf.drop_duplicates("zcta")[["zcta", "geometry", "lon", "lat"]]
        snap = gpd.GeoDataFrame(base.merge(agg, on="zcta"), geometry="geometry", crs=gdf.crs)
        label = "all-time"
    else:
        snap = gdf[gdf["month"] == pd.Timestamp(month)].copy()
        if snap.empty:
            raise ValueError(f"no rows for month {month}")
        label = pd.Timestamp(month).strftime("%Y-%m")
    dem_label = ACS_LABELS.get(acs_var, acs_var)

    def fmt(v):
        return "n/a" if pd.isna(v) else (f"{v:,.1f}" if isinstance(v, (float, np.floating)) else f"{v}")
    def loc(z, c, st):
        cs = ", ".join([x for x in [c if isinstance(c, str) and c else None,
                                    st if isinstance(st, str) and st else None] if x])
        return f"ZCTA {z} — {cs}" if cs else f"ZCTA {z}"
    def pwsline(p):
        if not isinstance(p, str) or not p:
            return ""
        parts = [x.strip() for x in p.split(";") if x.strip()]
        return "\nSystems: " + "; ".join(parts[:3]) + (f" +{len(parts)-3} more" if len(parts) > 3 else "")

    # ---- base layer: EVERY ZCTA outline; never-sampled areas shaded pink ----
    data_z = set(snap["zcta"])
    na = outline[~outline["zcta"].isin(data_z)].copy()
    na["fill_color"] = [NA_PINK + [200]] * len(na)
    na["tip"] = [f"ZCTA {z} — Unknown\n{dem_label}: Data Missing\nPFAS mean: Never sampled"
                 for z in na["zcta"]]
    na_layer = pdk.Layer(
        "GeoJsonLayer", json.loads(na[["zcta", "tip", "fill_color", "geometry"]].to_json()),
        get_fill_color="properties.fill_color", get_line_color=[70, 70, 80],
        line_width_min_pixels=0.2, stroked=True, filled=True, extruded=False, pickable=True)

    # ---- choropleth: data ZCTAs; a missing demographic is also pink ("Data Missing") ----
    dvals = pd.to_numeric(snap[acs_var], errors="coerce")
    dcol = ramp(norm(dvals))
    snap["fill_color"] = [NA_PINK + [alpha] if pd.isna(v) else [int(r), int(g), int(b), alpha]
                          for v, (r, g, b) in zip(dvals, dcol)]
    csv = snap["city"] if "city" in snap.columns else [None] * len(snap)
    ssv = snap["state"] if "state" in snap.columns else [None] * len(snap)
    psv = snap["pws_names"] if "pws_names" in snap.columns else [None] * len(snap)
    def demtxt(v):
        return "Data Missing" if pd.isna(v) else fmt(v)
    snap["tip"] = [loc(z, c, st) + f"\n{dem_label}: {demtxt(d)}\nPFAS mean: {fmt(p)} ng/L" + pwsline(pw)
                   for z, c, st, d, p, pw in
                   zip(snap["zcta"], csv, ssv, dvals, snap["mean_result_ngL"], psv)]
    choropleth = pdk.Layer(
        "GeoJsonLayer", json.loads(snap[["zcta", "tip", "fill_color", "geometry"]].to_json()),
        get_fill_color="properties.fill_color", get_line_color=[60, 60, 70], auto_highlight=True,
        line_width_min_pixels=0.2, stroked=True, filled=True, extruded=False, pickable=True)

    # thin 3D columns from PFAS (only where mean > 0; non-detects are 0 and would be flat)
    bars = snap[snap["mean_result_ngL"] > 0].copy()
    pn = norm(bars["mean_result_ngL"], hi_q=0.995)
    bcol = ramp(pn)
    bars["bar_color"] = [[int(r), int(g), int(b), 230] for r, g, b in bcol]
    bars["elev"] = (pn * elevation_max).fillna(0)
    columns = pdk.Layer(
        "ColumnLayer", bars[["lon", "lat", "elev", "bar_color", "tip", "zcta", "mean_result_ngL"]],
        get_position=["lon", "lat"], get_elevation="elev", elevation_scale=1,
        radius=radius, get_fill_color="bar_color", pickable=True, auto_highlight=True)

    # red ring + light-pink fill over top EJ-coincidence ZCTAs — ALWAYS shown
    bars["score"] = norm(bars[acs_var]).fillna(0).values * pn.values
    hs = bars.nlargest(n_hotspots, "score")
    halos = [pdk.Layer(
        "ScatterplotLayer", hs[["lon", "lat"]], get_position=["lon", "lat"],
        filled=True, stroked=True,
        get_fill_color=[255, 190, 200, 210],
        get_line_color=[210, 20, 20, 230], line_width_min_pixels=2.5,
        radius_min_pixels=16, radius_max_pixels=16, pickable=False)]

    # assemble; the pink base + choropleth go together so every area is shaded
    active = []
    if layers in ("both", "choropleth"):
        active += [na_layer, choropleth]
    if layers in ("both", "bars"):
        active.append(columns)
    active += halos

    view = pdk.ViewState(latitude=39.5, longitude=-98.5, zoom=4, pitch=86, bearing=0)
    map_view = pdk.View("MapView", controller={"dragRotate": True, "touchRotate": True})
    deck = pdk.Deck(layers=active, initial_view_state=view, views=[map_view],
                    map_provider="carto", map_style="light",
                    tooltip={"text": "{tip}"})

    if save:
        out = OUTDIR / f"map_{label}_{acs_var}_{layers}.html"
        deck.to_html(str(out), notebook_display=False)
        print(f"{label} [{layers}]: {len(active)} layers -> {out.name} "
              f"({out.stat().st_size/1e6:.1f} MB)")
    return deck

## Web integration

Exports the files the client-side deck.gl map loads once: `zcta.topojson` (sampled polygons),
`values.json` (per-ZCTA demographics + rurality, per-month PFAS mean/pws), `meta.json`
(vars, labels, months, rurality levels), and `zcta_all.topojson` (every US ZCTA outline,
carrying its `zcta` code so the map can shade never-sampled polygons).

`mean_result_ngL` includes non-detects as 0 (0 = sampled-clean; absent = never sampled).

In [134]:
# === export: geometry once (TopoJSON) + per-month values (Option 1) ===
# one-time: pip install topojson
import json, numpy as np, topojson
from pathlib import Path

WEB = Path(os.environ.get("WEB", "../web"))
WEB.mkdir(parents=True, exist_ok=True)

gdf = gdf[gdf["month"] >= pd.Timestamp("2023-02-01")].copy()

def _union(s):
    out = set()
    for v in s.dropna():
        out.update(p.strip() for p in str(v).split(";") if p.strip())
    return "; ".join(sorted(out))
def rnd(x, n=2):
    return None if pd.isna(x) else round(float(x), n)

# 1) sampled ZCTA polygons -> TopoJSON (shared once by every view)
geo_u = gdf.drop_duplicates("zcta")[["zcta", "geometry"]].reset_index(drop=True)
topo = topojson.Topology(geo_u, object_name="zctas", prequantize=1e5)
(WEB / "zcta.topojson").write_text(topo.to_json())

# 2) values nested by month: constant fields (incl. rurality) at ZCTA level, mean/pws per month
lonlat = gdf.groupby("zcta")[["lon", "lat"]].first()
has_city  = "city" in gdf.columns; has_state = "state" in gdf.columns
has_pws   = "pws_names" in gdf.columns; has_rural = "rurality" in gdf.columns
values = {}
for z, g in gdf.groupby("zcta"):
    rec = {c: rnd(g[c].iloc[0]) for c in ACS_VARS}
    if has_city:  rec["city"]  = g["city"].iloc[0]
    if has_state: rec["state"] = g["state"].iloc[0]
    if has_rural: rec["rurality"] = g["rurality"].iloc[0]
    rec["lon"] = rnd(lonlat.loc[z, "lon"], 4); rec["lat"] = rnd(lonlat.loc[z, "lat"], 4)
    months = {}
    for _, row in g.iterrows():
        if pd.isna(row["month"]):
            continue
        mk = pd.Timestamp(row["month"]).strftime("%Y-%m")
        entry = {"mean": rnd(row["mean_result_ngL"])}
        if has_pws: entry["pws"] = row["pws_names"]
        months[mk] = entry
    mm = [e["mean"] for e in months.values() if e["mean"] is not None]
    allrec = {"mean": rnd(np.mean(mm)) if mm else None}
    if has_pws: allrec["pws"] = _union(g["pws_names"])
    months["all"] = allrec
    rec["months"] = months
    values[z] = rec
(WEB / "values.json").write_text(json.dumps(values, separators=(",", ":")))

# 3) meta: variable list + labels + sorted months ("all" first) + rurality levels
all_months = sorted({pd.Timestamp(m).strftime("%Y-%m") for m in gdf["month"].dropna().unique()})
meta = {
    "acs_vars": ACS_VARS,
    "labels": {c: ACS_LABELS.get(c, c) for c in ACS_VARS},
    "months": ["all"] + all_months,
    "rurality_levels": ["Rural", "Suburban", "Urban", "Unknown"],
}
(WEB / "meta.json").write_text(json.dumps(meta))

print(f"zcta.topojson: {(WEB/'zcta.topojson').stat().st_size/1e6:.2f} MB ({len(geo_u):,} ZCTAs)")
print(f"values.json  : {(WEB/'values.json').stat().st_size/1e6:.2f} MB")
print(f"vars exported: {ACS_VARS}")
print(f"months: {len(all_months)} ({all_months[0]} .. {all_months[-1]})")

zcta.topojson: 4.62 MB (18,191 ZCTAs)
values.json  : 12.23 MB
vars exported: ['total_pop', 'median_hh_income', 'pct_below_poverty', 'pct_nonwhite', 'pct_hispanic', 'pct_black_nh', 'pct_asian_nh', 'pop_density']
months: 35 (2023-02 .. 2025-12)


In [135]:
# === one-time: ALL-ZCTA outline + N/A-fill base for the WEB map (loaded once) ===
# keeps the ZCTA code so the map can shade polygons with no data as "never sampled"
import geopandas as gpd, topojson
allz = gpd.read_file(TIGER)
zcol = next(c for c in allz.columns if c.upper().startswith("ZCTA5CE"))
allz = allz[[zcol, "geometry"]].rename(columns={zcol: "zcta"}).to_crs(4326)
allz["zcta"] = allz["zcta"].astype(str).str.zfill(5)
allz["geometry"] = allz.simplify(0.02, preserve_topology=True)   # coarse — thin outlines / N/A fill
topo_all = topojson.Topology(allz[["zcta", "geometry"]], object_name="zcta_all", prequantize=1e5)
(WEB / "zcta_all.topojson").write_text(topo_all.to_json())
print(f"zcta_all.topojson: {(WEB/'zcta_all.topojson').stat().st_size/1e6:.1f} MB ({len(allz):,} ZCTAs)")

zcta_all.topojson: 6.4 MB (33,791 ZCTAs)
